# HOT-Step CPP on Google Colab

Runs the HOT-Step Linux release on a Colab GPU and gives you a link to the web UI.

Run the cells in order. Change the settings in each cell's form before running it.

1. **Install**: downloads the release and the CUDA 12 libraries the engine needs.
2. **Google Drive** (optional): keeps models, adapters and your songs between sessions.
3. **Models**: downloads a model pack. You can add more later from the Model Manager in the UI.
4. **Start**: launches the server and waits until it answers.
5. **Open the app**: prints the link, through Colab's own proxy or ngrok.
6. **Back up** (Drive only): copies your songs and training data to Drive. Run it before the session ends.

A free T4 runs ACE-Step and YuE2. Prefer Q8_0 models on a T4: it has no fast BF16 path.

Docs: https://github.com/scragnog/HOT-Step-CPP/blob/master/docs/user/colab.md

In [ ]:
#@title 1. Install HOT-Step
VERSION = "latest"  #@param {type:"string"}
#@markdown `latest`, or a tag such as `v1.3.6`.

import json, os, subprocess, sys, urllib.request

ROOT = "/content/hotstep"
ASSET_SUFFIX = "linux-x64-cuda12.8.tar.gz"
CUDA_LIBS = "/content/cuda12"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > pick a GPU, then run this cell again.")
print("GPU:", gpu.stdout.strip())

api = "https://api.github.com/repos/scragnog/HOT-Step-CPP/releases/"
api += "latest" if VERSION.strip() in ("", "latest") else "tags/" + VERSION.strip()
release = json.load(urllib.request.urlopen(api))
asset = next((a for a in release["assets"] if a["name"].endswith(ASSET_SUFFIX)), None)
if asset is None:
    raise SystemExit(f"Release {release['tag_name']} has no *{ASSET_SUFFIX} asset.")

# Names in server/data/ that ship with the release. That folder also holds your library,
# and cell 6 leaves these out of the backup so a restore never rolls them back.
RELEASE_DATA = f"{ROOT}/.release-data-files"

installed = open(f"{ROOT}/VERSION").read().strip() if os.path.exists(f"{ROOT}/VERSION") else None
if installed == release["tag_name"].lstrip("v") and os.path.exists(RELEASE_DATA):
    print(f"{installed} is already installed.")
else:
    print(f"Downloading {asset['name']} ({asset['size'] / 1e6:.0f} MB)")
    subprocess.run(["wget", "-q", "--show-progress", "-O", "/content/hotstep.tar.gz",
                    asset["browser_download_url"]], check=True)
    os.makedirs(ROOT, exist_ok=True)
    # Only program files are replaced: models, adapters and data survive a reinstall,
    # and --keep-directory-symlink keeps the Drive links from cell 2.
    subprocess.run(["tar", "-xzf", "/content/hotstep.tar.gz", "-C", ROOT, "--strip-components=1",
                    "--keep-directory-symlink"], check=True)
    listing = subprocess.run(["tar", "-tzf", "/content/hotstep.tar.gz"],
                             capture_output=True, text=True, check=True).stdout.splitlines()
    parts = [(p[2:] if p.startswith("./") else p).split("/") for p in listing]
    names = {q[3] for q in parts if len(q) > 3 and q[1:3] == ["server", "data"] and q[3]}
    # Leading "/" anchors each rsync exclude to the top of server/data.
    open(RELEASE_DATA, "w").write("".join(f"/{n}\n" for n in sorted(names)))
    os.remove("/content/hotstep.tar.gz")
    print("Installed", open(f"{ROOT}/VERSION").read().strip())

# The release does not bundle the CUDA runtime; on Linux it comes from the system.
# Colab's image does not always carry the CUDA 12 libraries, so install them into
# their own folder (leaving Colab's PyTorch alone) when the engine cannot find them.
def missing_libs():
    # Keep Colab's own LD_LIBRARY_PATH: the driver's libcuda.so.1 lives in /usr/lib64-nvidia.
    path = [f"{ROOT}/engine"] + cuda_lib_dirs() + [os.environ.get("LD_LIBRARY_PATH", "")]
    env = {**os.environ, "LD_LIBRARY_PATH": ":".join(path)}
    # Only what generation loads: ace-server and the ggml backends. Other .so files in
    # engine/ are optional extras (TensorRT and similar) whose libraries Colab lacks.
    engine = f"{ROOT}/engine"
    targets = [f"{engine}/ace-server"] + [f"{engine}/{n}" for n in os.listdir(engine)
                                          if "ggml" in n and n.endswith(".so")]
    out = ""
    for f in targets:
        out += subprocess.run(["ldd", f], capture_output=True, text=True, env=env).stdout
    return sorted({l.split()[0] for l in out.splitlines() if "not found" in l})

def cuda_lib_dirs():
    base = f"{CUDA_LIBS}/nvidia"
    return [f"{base}/{d}/lib" for d in ("cuda_runtime", "cublas")] if os.path.isdir(base) else []

if missing_libs():
    print("Installing CUDA 12 runtime libraries:", ", ".join(missing_libs()))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", CUDA_LIBS,
                    "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"], check=True)
still = missing_libs()
if still:
    raise SystemExit("The engine still cannot find: " + ", ".join(still))
print("Engine libraries: all found.")

In [ ]:
#@title 2. Google Drive (optional)
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "HOT-Step"  #@param {type:"string"}
#@markdown Models and adapters are stored on Drive directly, so they download once.
#@markdown Songs and training data stay on the Colab disk while you work (the database
#@markdown does not run reliably on Drive); cell 6 copies them back.

import os, shutil

ROOT = "/content/hotstep"
DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for name in ("models", "adapters"):
        target, link = f"{DRIVE}/{name}", f"{ROOT}/{name}"
        os.makedirs(target, exist_ok=True)
        if os.path.islink(link):
            continue
        if os.path.isdir(link):
            # Move anything already downloaded this session onto Drive.
            for entry in os.listdir(link):
                if not os.path.exists(f"{target}/{entry}"):
                    shutil.move(f"{link}/{entry}", target)
            shutil.rmtree(link)
        os.symlink(target, link)
        print(f"{name}/ -> {target}")
    # The release keeps the library in server/data (DATA_DIR in .env).
    if os.path.exists(f"{DRIVE}/data/hotstep.db") and not os.path.exists(f"{ROOT}/server/data/hotstep.db"):
        print("Restoring songs and training data from Drive...")
        shutil.copytree(f"{DRIVE}/data", f"{ROOT}/server/data", dirs_exist_ok=True)
else:
    print("Drive skipped. Everything is lost when the session ends.")

In [ ]:
#@title 3. Download models
PACK = "yue2-recommended"  #@param ["quick-start", "minimal", "xl-quality", "yue2-recommended", "yue2-compact-imatrix", "yue2-full-precision", "none"]
HF_TOKEN = ""  #@param {type:"string"}
#@markdown Packs come from the app's own model registry, the same list the Model Manager shows.
#@markdown `quick-start` / `minimal` / `xl-quality` are ACE-Step 1.5; the `yue2-*` packs are YuE2.
#@markdown The HF token is optional; it only speeds up downloads.

import json, os, shutil
from huggingface_hub import hf_hub_download

ROOT = "/content/hotstep"
MODELS = f"{ROOT}/models"

registry = json.load(open(f"{ROOT}/server/data/model-registry.json"))
files = {f["id"]: f for f in registry["files"]}

def fetch(repo, repo_path, dest, size=None):
    # A size mismatch means a copy cut off by an earlier session: fetch it again.
    if os.path.exists(dest) and (size is None or os.path.getsize(dest) == size):
        print("  have", os.path.basename(dest))
        return
    print("  downloading", os.path.basename(dest))
    tmp = f"/content/hf-tmp"
    got = hf_hub_download(repo, repo_path, local_dir=tmp, token=HF_TOKEN.strip() or None)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.move(got, dest)
    shutil.rmtree(tmp, ignore_errors=True)

if PACK != "none":
    pack = next(p for p in registry["packs"] if p["id"] == PACK)
    print(pack["name"])
    for fid in pack["fileIds"]:
        f = files[fid]
        if f["filename"].lower().endswith(".dll"):
            continue  # Windows CUDA runtime; cell 1 handles Linux
        folder = os.path.join(MODELS, f.get("subdir", ""))
        fetch(f["repo"], f.get("repoPath", f["filename"]), os.path.join(folder, f["filename"]),
              f.get("sizeBytes"))
        for c in f.get("companions", []):
            fetch(f["repo"], c["repoPath"], os.path.join(folder, c["filename"]))
    print("Done.")

In [ ]:
#@title 4. Start HOT-Step
import json, os, subprocess, time, urllib.request

ROOT = "/content/hotstep"
LOG = f"{ROOT}/colab-server.log"
CUDA_LIBS = "/content/cuda12/nvidia"

for pattern in ("HOT-Step.sh", "server/server.mjs", "engine/ace-server"):
    subprocess.run(["pkill", "-f", pattern])
time.sleep(2)

lib_dirs = [f"{ROOT}/engine"] + [f"{CUDA_LIBS}/{d}/lib" for d in ("cuda_runtime", "cublas")
                                 if os.path.isdir(f"{CUDA_LIBS}/{d}/lib")]
env = {**os.environ, "HOT_STEP_ROOT": ROOT,
       "LD_LIBRARY_PATH": ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])}
# HOT-Step.sh restarts the server when you press Restart in Settings.
subprocess.Popen(["bash", f"{ROOT}/HOT-Step.sh"], cwd=ROOT, env=env,
                 stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

print("Starting", end="", flush=True)
health = None
for _ in range(180):
    time.sleep(2)
    print(".", end="", flush=True)
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:3001/api/health", timeout=3))
        if health["engine"]["ready"] and health["aceServer"]["status"] != "disconnected":
            break
    except Exception:
        pass
    if "giving up" in open(LOG).read():
        break  # the server stopped respawning a crashing engine
print()

if health and health["aceServer"]["status"] != "disconnected":
    print(f"HOT-Step {health['version']} is running. Run cell 5 for the URL.")
else:
    print("The server did not come up. Last lines of the log:\n")
    print("".join(open(LOG).readlines()[-40:]))

In [ ]:
#@title 5. Open the app
TUNNEL = "colab"  #@param ["colab", "ngrok"]
NGROK_TOKEN = ""  #@param {type:"string"}
#@markdown `colab`: Colab's own port proxy. No account needed, and the link only opens in a
#@markdown browser signed in to this Google account.
#@markdown `ngrok`: a public link you can open anywhere. Needs a free authtoken from
#@markdown https://dashboard.ngrok.com/get-started/your-authtoken. Anyone with the link can
#@markdown use your session, so don't share it.
#@markdown (Cloudflare quick tunnels are not offered: they drop the live streams the
#@markdown terminal, downloads and assistant rely on.)
import subprocess, sys

if TUNNEL == "colab":
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(3001)")
else:
    if not NGROK_TOKEN.strip():
        raise SystemExit("Paste your ngrok authtoken into NGROK_TOKEN first.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    ngrok.kill()
    url = ngrok.connect(3001, "http").public_url
print("Open HOT-Step at:", url)

In [ ]:
#@title 6. Back up songs and training data to Drive
#@markdown Copies `data/` (songs, library database, training datasets and runs) to Drive.
#@markdown Cell 2 restores it next session. Stop any running generation or training first.
import os, shutil, sqlite3, subprocess

ROOT = "/content/hotstep"
DRIVE = f"/content/drive/MyDrive/{globals().get('DRIVE_FOLDER', 'HOT-Step')}"
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("Drive is not mounted. Run cell 2 with USE_DRIVE ticked.")
os.makedirs(f"{DRIVE}/data", exist_ok=True)

# Contents and timestamps only: Drive rejects ownership and permission changes,
# which is what `rsync -a` tries. The live database is left to the snapshot below.
# The release keeps the library in server/data (DATA_DIR in .env), next to files that
# ship with it; cell 1 listed those so they stay out of the backup.
DATA = f"{ROOT}/server/data"
if not os.path.exists(f"{ROOT}/.release-data-files"):
    raise SystemExit("Run cell 1 first: it records which files in server/data come with the release.")
r = subprocess.run(["rsync", "-rt", "--info=stats1", "--exclude", "hotstep.db*",
                    "--exclude-from", f"{ROOT}/.release-data-files",
                    f"{DATA}/", f"{DRIVE}/data/"], capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    raise SystemExit("rsync failed (exit %d):\n%s" % (r.returncode, r.stderr[-2000:]))

# A plain copy of a database the server has open can catch it mid-write; SQLite's
# backup API takes a consistent snapshot instead.
db = f"{DATA}/hotstep.db"
if os.path.exists(db):
    src, snap = sqlite3.connect(db), sqlite3.connect("/content/hotstep-db-snapshot.db")
    src.backup(snap)
    src.close(); snap.close()
    shutil.copyfile("/content/hotstep-db-snapshot.db", f"{DRIVE}/data/hotstep.db")
    os.remove("/content/hotstep-db-snapshot.db")
print("Backed up to", f"{DRIVE}/data")